# Implement FSDP (Fully Sharded Data Parallel) from Scratch

**Difficulty**: 🟣 Expert

**Companies**: Meta, Google, NVIDIA, Anthropic, xAI

---

### Problem Statement

Training large models requires distributing them across multiple GPUs. **Fully Sharded Data Parallel (FSDP)** is the state-of-the-art approach that shards model parameters, gradients, and optimizer states across all ranks (GPUs). This is based on Microsoft's **ZeRO** (Zero Redundancy Optimizer) paper.

The key insight: in standard Data Parallel (DDP), every GPU holds a **full copy** of the model. With FSDP:
- **At rest**: each rank holds only `1/world_size` of each parameter (a "shard")
- **Before forward**: parameters are **all-gathered** so each rank has the full parameter temporarily
- **After forward**: full parameters are discarded, keeping only the shard
- **After backward**: gradients are **reduce-scattered** so each rank gets the gradient for its shard only

This reduces per-GPU memory from `O(model)` to `O(model / world_size)`.

Since we cannot assume multi-GPU hardware, you will simulate distributed operations using a `FakeDistributed` class.

---

### Requirements

1. **FakeDistributed** — Simulate `world_size` ranks with `all_gather` and `reduce_scatter` on CPU.
2. **FSDPLayer** — An `nn.Module` wrapper that shards weights, all-gathers for forward, and reduce-scatters gradients.
3. **Three Strategies** — Implement Full Shard (ZeRO-3), Shard Grad+Optimizer (ZeRO-2), and No Shard (DDP).
4. **Validation** — Forward output must be identical regardless of sharding. Gradients must match single-device training.

---

### Constraints

- ✅ All computation happens on CPU using simulated distributed ops
- ✅ Must demonstrate correct gradient computation
- ✅ Must show per-rank memory usage for each strategy
- ❌ Do **not** use `torch.distributed` (we simulate everything)

---

<details>
  <summary>💡 Hint</summary>

  **Simulating distributed operations:**
  ```
  all_gather: Given shards [s0, s1, s2, s3], return full tensor = cat([s0, s1, s2, s3])
  reduce_scatter: Given full grads [g0, g1, g2, g3], 
                  split into chunks, sum across ranks, return shard for each rank
  ```
  
  **FSDP forward pass:**
  1. All-gather the parameter shards to reconstruct full weight
  2. Perform the forward computation with the full weight
  3. (Optionally) discard the full weight to save memory
  
  **FSDP backward pass:**
  1. All-gather parameters again (needed for gradient computation)
  2. Compute gradients w.r.t. the full parameter
  3. Reduce-scatter gradients: each rank gets gradient for its shard only

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Optional
from enum import Enum
import math

In [ ]:
# Setup
torch.manual_seed(42)

# Simulated distributed configuration
WORLD_SIZE = 4  # simulate 4 GPUs

# Model dimensions
INPUT_DIM = 16
HIDDEN_DIM = 32
OUTPUT_DIM = 8
BATCH_SIZE = 4

print(f"Simulating {WORLD_SIZE} GPUs")
print(f"Model: {INPUT_DIM} -> {HIDDEN_DIM} -> {OUTPUT_DIM}")
print(f"Batch size: {BATCH_SIZE}")

In [ ]:
class ShardingStrategy(Enum):
    """FSDP sharding strategies corresponding to ZeRO stages."""
    FULL_SHARD = "full_shard"        # ZeRO-3: shard params + grads + optimizer states
    SHARD_GRAD_OP = "shard_grad_op"  # ZeRO-2: shard grads + optimizer states only
    NO_SHARD = "no_shard"            # DDP: replicate everything


class FakeDistributed:
    """
    Simulates torch.distributed operations on CPU.
    Instead of actual inter-GPU communication, we manipulate lists of tensors.
    """
    
    def __init__(self, world_size: int):
        self.world_size = world_size
    
    def shard(self, tensor: torch.Tensor, dim: int = 0) -> List[torch.Tensor]:
        """
        Split a tensor into world_size equal shards along dim.
        
        Args:
            tensor: full parameter tensor
            dim: dimension to shard along
        Returns:
            List of world_size shard tensors
        """
        # TODO: Split tensor into world_size chunks along dim
        # Use torch.chunk(tensor, self.world_size, dim=dim)
        # Return list of detached clones (each rank has its own copy)
        ...
    
    def all_gather(self, shards: List[torch.Tensor], dim: int = 0) -> torch.Tensor:
        """
        Simulate all-gather: concatenate shards from all ranks.
        In real distributed training, each rank sends its shard to all other ranks.
        
        Args:
            shards: list of tensors, one per rank
            dim: dimension to concatenate along
        Returns:
            Full tensor reconstructed from all shards
        """
        # TODO: Concatenate all shards along dim
        ...
    
    def reduce_scatter(self, full_grads: List[torch.Tensor], dim: int = 0) -> List[torch.Tensor]:
        """
        Simulate reduce-scatter: sum gradients across ranks, then scatter shards.
        Each rank ends up with the reduced gradient for its shard only.
        
        In real training:
        1. Each rank has the full gradient (computed from its local data)
        2. Gradients are summed across ranks (reduce)
        3. The summed gradient is split, each rank gets its shard (scatter)
        
        Args:
            full_grads: list of full gradient tensors, one per rank
            dim: dimension to shard along
        Returns:
            List of gradient shards, one per rank
        """
        # TODO: 
        # 1. Sum all full gradients element-wise: summed = sum(full_grads)
        # 2. Chunk the summed gradient into world_size shards
        # 3. Return list of shards (rank i gets chunk i)
        ...

print("FakeDistributed class defined.")

In [ ]:
class FSDPLinear(nn.Module):
    """
    A linear layer wrapped with FSDP-style sharding.
    
    The weight matrix is sharded across ranks along the output dimension.
    Before forward: all-gather to reconstruct the full weight.
    After backward: reduce-scatter the gradients.
    """
    
    def __init__(
        self,
        in_features: int,
        out_features: int,
        dist: FakeDistributed,
        strategy: ShardingStrategy = ShardingStrategy.FULL_SHARD,
    ):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.dist = dist
        self.strategy = strategy
        self.world_size = dist.world_size
        
        # TODO: Create the full weight and bias, then shard them
        # 1. Create full_weight of shape (out_features, in_features) with Xavier init
        # 2. Create full_bias of shape (out_features,)
        # 3. Shard the weight along dim=0 (output dimension)
        # 4. Shard the bias along dim=0
        # 5. Store shards as a list: self.weight_shards
        # 6. Store the full weight for reference (single-device comparison)
        ...
    
    def forward_on_rank(self, x: torch.Tensor, rank: int) -> torch.Tensor:
        """
        Forward pass as seen by a specific rank.
        
        Args:
            x: input tensor (batch_size, in_features)
            rank: which simulated GPU is running this
        Returns:
            output: (batch_size, out_features)
        """
        # TODO:
        # 1. All-gather weight shards to reconstruct full weight
        # 2. All-gather bias shards to reconstruct full bias
        # 3. Compute F.linear(x, full_weight, full_bias)
        # 4. If strategy is FULL_SHARD, note that we'd discard full_weight after
        #    (in practice, this saves memory; here we just compute and return)
        ...
    
    def compute_gradient_shards(self, x: torch.Tensor, grad_output: torch.Tensor) -> List[torch.Tensor]:
        """
        Simulate gradient computation and reduce-scatter.
        
        In real FSDP:
        1. Each rank computes the full gradient from its local batch
        2. Reduce-scatter gives each rank its gradient shard
        
        Args:
            x: input tensor (batch_size, in_features)
            grad_output: gradient of loss w.r.t. output (batch_size, out_features)
        Returns:
            List of gradient shards, one per rank
        """
        # TODO:
        # 1. Compute full weight gradient: grad_weight = grad_output^T @ x
        #    Shape: (out_features, in_features)
        # 2. Simulate each rank computing the same gradient (from different data
        #    batches in practice, but we use the same for simplicity)
        # 3. Reduce-scatter to get per-rank gradient shards
        ...
    
    def memory_per_rank(self) -> dict:
        """
        Report memory usage per rank for this layer under current strategy.
        Returns dict with parameter and gradient memory in elements.
        """
        # TODO: Compute memory based on strategy
        # FULL_SHARD: params = total/world_size, grads = total/world_size
        # SHARD_GRAD_OP: params = total (kept full), grads = total/world_size
        # NO_SHARD: params = total, grads = total
        ...

print("FSDPLinear class defined.")

In [ ]:
# Validation
print("=" * 60)
print("VALIDATING FSDP IMPLEMENTATION")
print("=" * 60)

dist = FakeDistributed(world_size=WORLD_SIZE)

# Create test input
torch.manual_seed(42)
x = torch.randn(BATCH_SIZE, INPUT_DIM)

# Test 1: All-gather reconstructs the original tensor
print("\n--- Test 1: All-Gather Reconstruction ---")
test_tensor = torch.randn(HIDDEN_DIM, INPUT_DIM)
shards = dist.shard(test_tensor, dim=0)
print(f"Original shape: {test_tensor.shape}")
print(f"Shard shapes: {[s.shape for s in shards]}")
reconstructed = dist.all_gather(shards, dim=0)
print(f"Reconstructed shape: {reconstructed.shape}")
assert torch.allclose(test_tensor, reconstructed), "All-gather reconstruction FAILED!"
print("PASSED")

# Test 2: Reduce-scatter produces correct shards
print("\n--- Test 2: Reduce-Scatter ---")
grad1 = torch.randn(HIDDEN_DIM, INPUT_DIM)
grad2 = torch.randn(HIDDEN_DIM, INPUT_DIM)
grad3 = torch.randn(HIDDEN_DIM, INPUT_DIM)
grad4 = torch.randn(HIDDEN_DIM, INPUT_DIM)
result_shards = dist.reduce_scatter([grad1, grad2, grad3, grad4], dim=0)
summed = grad1 + grad2 + grad3 + grad4
expected_shards = torch.chunk(summed, WORLD_SIZE, dim=0)
for i, (got, expected) in enumerate(zip(result_shards, expected_shards)):
    assert torch.allclose(got, expected), f"Reduce-scatter shard {i} FAILED!"
print(f"Reduce-scatter produced {len(result_shards)} correct shards")
print("PASSED")

# Test 3: Forward pass produces same output regardless of rank
print("\n--- Test 3: Forward Pass Consistency ---")
layer = FSDPLinear(INPUT_DIM, HIDDEN_DIM, dist, ShardingStrategy.FULL_SHARD)
outputs = []
for rank in range(WORLD_SIZE):
    out = layer.forward_on_rank(x, rank)
    outputs.append(out)
    print(f"  Rank {rank} output shape: {out.shape}")

for i in range(1, WORLD_SIZE):
    assert torch.allclose(outputs[0], outputs[i], atol=1e-6), f"Rank {i} output differs!"
print("All ranks produce identical output. PASSED")

# Test 4: Forward matches single-device computation
print("\n--- Test 4: Correctness vs Single-Device ---")
single_output = F.linear(x, layer.full_weight, layer.full_bias)
fsdp_output = outputs[0]
max_err = (fsdp_output - single_output).abs().max().item()
print(f"Max error vs single-device: {max_err:.2e}")
assert torch.allclose(fsdp_output, single_output, atol=1e-6), "FSDP output != single-device!"
print("PASSED")

# Test 5: Gradient reduce-scatter
print("\n--- Test 5: Gradient Reduce-Scatter ---")
grad_output = torch.randn(BATCH_SIZE, HIDDEN_DIM)
grad_shards = layer.compute_gradient_shards(x, grad_output)
print(f"Gradient shard shapes: {[s.shape for s in grad_shards]}")
full_grad = dist.all_gather(grad_shards, dim=0)
expected_grad = grad_output.T @ x  # single-device gradient
# The reduce-scattered gradient (when gathered) should be world_size * single gradient
# because each rank computes the same gradient and they get summed
print(f"Full gathered gradient shape: {full_grad.shape}")
print("PASSED")

# Test 6: Memory comparison across strategies
print("\n--- Test 6: Memory Comparison ---")
for strategy in ShardingStrategy:
    layer_s = FSDPLinear(INPUT_DIM, HIDDEN_DIM, dist, strategy)
    mem = layer_s.memory_per_rank()
    print(f"  {strategy.value:>15s}: params={mem['params']:>5d}, grads={mem['grads']:>5d} elements/rank")

print("\nAll tests passed!")